# 🚀 Etapa 1: Análise Exploratória e Carga de Dados

Nesta etapa inicial, carregamos os dados brutos disponibilizados para o desafio. O objetivo principal é compreender a estrutura do dataset de treinamento, que contém características dos estabelecimentos (como localização, atributos e categorias) e a nossa variável alvo: `destaque` (1 para alta avaliação, 0 caso contrário).

## Dataset original

In [1]:
## 🛠 Engenharia de Features Avançada

Para obter um desempenho excelente na predição de `destaque`, aplicamos um pipeline de engenharia de features profundo, combinando técnicas geoespaciais, de NLP e de redução de dimensionalidade.

### 1. Tratamento da coluna `categories` (NLP e PCA)
- **Vetorização Semântica (Word Embeddings)**: Diferente de abordagens tradicionais, usamos modelos pré-treinados do spaCy (`en_core_web_md`) para gerar vetores semânticos médios das categorias, capturando o "significado" do tipo de negócio.
- **Similaridade de Cosseno**: Comparamos os vetores de estabelecimentos com perfis historicamente mal avaliados para criar uma "taxa de similaridade" (`categories_cossine`).
- **Redução de Dimensionalidade (PCA)**: As categorias também sofreram *One-Hot Encoding*. Para evitar a maldição da dimensionalidade, aplicamos o **PCA (Principal Component Analysis)**, resumindo centenas de categorias em apenas 3 componentes principais.

### 2. Tratamento da Coluna `attributes`
- O JSON complexo de atributos do dataset original (e.g. `WiFi`, `NoiseLevel`) foi normalizado. 
- Aplicamos PCA para extrair as variáveis latentes mais explicativas sobre as comodidades oferecidas pelos estabelecimentos.

### 3. Clusterização Geoespacial com DBSCAN
- **Por que DBSCAN?** Latitude e longitude puras não têm relação linear forte com a avaliação. Aplicamos DBSCAN para descobrir clusters de densidade (polos comerciais populares e não-populares) independentemente da sua forma geométrica.
- O algoritmo converteu a localização física em *features* indicadoras de presença em zonas quentes da cidade de Toronto.

### 4. Análise de Sentimentos (Reviews)
- Consolidamos centenas de milhares de *reviews* usando a ferramenta de NLP NLTK (`SentimentIntensityAnalyzer`). Extraímos a polaridade composta (`compound`) para gerar um score contínuo refletindo a satisfação média dos clientes.


INFO:root:Carregando dados


'Tamanho dataset: 17582'

,business_id,name,address,postal_code,latitude,longitude,review_count,is_open,attributes,categories,hours,loc,destaque
0,vHzWmPWHN4J1hRR3W3AMQg,Salt Wine Bar,225 Ossington Ave,M6J 2Z8,43.648977,-79.420495,99,1,"{'Ambience': ""{'romantic': False, 'intimate': ...","Wine Bars, Tapas/Small Plates, Restaurants, Ba...","{'Monday': '18:0-23:0', 'Tuesday': '18:0-23:0'...","{'type': 'Point', 'coordinates': [-79.4204946,...",0
1,15to24Q-otAHmto7FzsWRg,William's Beauty Supplies,2229 Dundas Street W,M6R 1X6,43.654002,-79.452189,3,1,"{'BusinessParking': ""{'garage': False, 'street...","Beauty & Spas, Hair Salons, Barbers, Shopping,...","{'Monday': '9:0-17:0', 'Tuesday': '9:0-17:0', ...","{'type': 'Point', 'coordinates': [-79.4521893,...",1
2,8aqKdf4G4AAir8k_Kdslvg,Integra Health Centre,1320-130 King Street W,M5X 1C8,43.648493,-79.383214,18,1,"{'ByAppointmentOnly': 'True', 'AcceptsInsuranc...","Health & Medical, Medical Centers","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ...","{'type': 'Point', 'coordinates': [-79.3832139,...",0
3,uxU1vr5AhhkTQ83X0bpeyg,North York General Hospital,555 Finch Avenue W,M2R 1N5,43.772453,-79.448136,3,0,{'ByAppointmentOnly': 'False'},"Health & Medical, Hospitals, Medical Centers",NaN,"{'type': 'Point', 'coordinates': [-79.4481361,...",0
4,f702hTJoqdR34Jn23C7d1A,Dr Jamie's Bike Clinic,2741 Dundas St W,M6P,43.665082,-79.460800,3,1,{'RestaurantsPriceRange2': '3'},"Automotive, Bikes, Shopping, Auto Repair, Spor...",NaN,"{'type': 'Point', 'coordinates': [-79.4607999,...",0


## Tratamento coluna `categories`

- **Tratamento de Linhas Vazias**: As linhas da coluna `categories` que estavam vazias ou nulas foram tratadas para evitar problemas na análise posterior.

- **Separação das Categorias**: As strings da coluna `categories`, contendo múltiplas categorias separadas por vírgulas, foram divididas em listas de categorias individuais.

- **Explosão das Listas de Categorias**: Após a separação, a coluna foi "explodida", ou seja, cada categoria foi expandida em uma linha separada, mantendo as demais colunas constantes. Isso permitiu um tratamento individualizado de cada categoria.

- **Codificação das Categorias**: As categorias foram codificadas em colunas booleanas (one-hot encoding), onde cada categoria se tornou uma coluna no DataFrame, com valores `0` ou `1` indicando a presença ou ausência daquela categoria.

- **Cálculo de Embeddings**: A coluna `categories` foi transformada em embeddings utilizando técnicas de processamento de linguagem natural (Word Embeddings), permitindo que as categorias sejam representadas por vetores numéricos que capturam semânticas.

- **Análise de Similaridade**: A similaridade entre as categorias foi calculada usando a similaridade do cosseno entre os embeddings, permitindo identificar quão parecidas são as categorias entre si.

- **Redução de Dimensionalidade**: Os embeddings das categorias passaram por uma redução de dimensionalidade utilizando PCA (Principal Component Analysis), resultando em componentes principais que foram utilizados como features no modelo final.

- **Identificação de Categorias Populares**: Uma coluna adicional foi criada para identificar se uma categoria pertence ao conjunto das categorias mais populares, auxiliando na criação de features mais informativas.

## Tratamento da Coluna `attributes`

- **Extração e Normalização**: A coluna `attributes`, que continha informações estruturadas como JSON em formato de string, foi convertida em um DataFrame utilizando `json_normalize`. Isso permitiu que cada atributo presente na string fosse separado em colunas distintas para facilitar a análise.

- **Tratamento de Valores Nulos**: Após a normalização, as colunas que resultaram da extração de atributos tiveram seus valores nulos preenchidos com a string `"SEM_VALOR"`. Isso foi necessário para garantir que modelos preditivos futuros não enfrentassem problemas com dados ausentes.

- **Processamento de Atributos Textuais**: Alguns atributos textuais, como `WiFi`, `NoiseLevel`, e `RestaurantsAttire`, passaram por um tratamento adicional para remover prefixos indesejados (e.g., `'u'`) e outros caracteres que poderiam interferir na análise.

## Agrupamento DBSCAN

- **Objetivo do Agrupamento**: O algoritmo de agrupamento DBSCAN foi aplicado para identificar clusters naturais entre os estabelecimentos com base em suas características numéricas. Isso foi útil para detectar padrões de comportamento entre os estabelecimentos, como agrupamentos de estabelecimentos com características semelhantes.

- **Parâmetros do Algoritmo**: Os parâmetros `eps` e `min_samples` foram ajustados para identificar agrupamentos densos e ignorar pontos ruidosos. O valor de `eps` define a distância máxima entre dois pontos para que eles sejam considerados parte do mesmo cluster, enquanto `min_samples` é o número mínimo de pontos necessários para formar um cluster.

- **Interpretação dos Resultados**: Os clusters resultantes foram interpretados para entender se havia algum padrão específico entre os estabelecimentos. Essa análise foi fundamental para categorizar os estabelecimentos antes de aplicar os modelos preditivos.

## Análise dos Reviews dos Estabelecimentos

- **Coleta e Limpeza dos Reviews**: As avaliações textuais dos estabelecimentos foram coletadas e passaram por um processo de limpeza.

- **Geração de Embeddings**: Foi utilizado um modelo de linguagem para converter os reviews em embeddings, representações vetoriais que capturam o significado semântico dos textos. Esses embeddings foram utilizados como insumos para a análise de sentimentos e também para treinar os modelos preditivos.

- **Integração com Atributos**: As características extraídas dos reviews foram integradas com as outras características dos estabelecimentos, como as presentes na coluna `attributes`, formando um conjunto de dados rico e completo para o treinamento dos modelos preditivos.

## Considerações Finais

- **Preparação para Modelos Preditivos**: O tratamento das features preparou o dataset para o treinamento de modelos preditivos. Essas etapas foram essenciais para garantir a qualidade e a relevância dos dados, resultando em um modelo mais robusto e preciso.


# 📊 Dataset Tratado e Padronizado
Após toda a transformação, o nosso espaço multidimensional foi escalonado usando `StandardScaler` e está pronto para alimentar os classificadores.

In [2]:
df_treino, df_validacao = carga.carregar_dados()

display(f"Tamanho dataset treinamento: {len(df_treino)}")

display(df_treino.head())

INFO:root:Carregando dados
INFO:root:Tratando linhas com categoria vazia
INFO:root:Seleção PCA para categorias
INFO:root:Tratando categorias não populares usando similaridade do cosseno
100%|██████████| 17582/17582 [07:45<00:00, 37.81it/s]
INFO:root:Adicionando coluna categoria popular
INFO:root:Acrescentando coluna embedding para representar as categorias
100%|██████████| 17582/17582 [01:46<00:00, 164.90it/s]
INFO:root:Tratando reviews
100%|██████████| 490963/490963 [08:45<00:00, 934.77it/s] 
INFO:root:Padronizar dados


'Tamanho dataset treinamento: 14065'

,review_count,is_open,cat_pca_0,cat_pca_1,cat_pca_2,categories_cossine,popular_categories,categories_embedding,attr_pca_0,attr_pca_1,attr_pca_2,agr_lat_log_popular_0.0,agr_lat_log_popular_1.0,agr_lat_log_popular_2.0,agr_lat_log_nao_popular_0.0,avg_sentimento,destaque
business_id,,,,,,,,,,,,,,,,,
vHzWmPWHN4J1hRR3W3AMQg,1.190565,0.564871,0.888214,-0.636469,0.071289,0.105376,-0.25532,-0.434373,-0.296532,0.018263,-0.172969,2.408640,-0.04715,-0.016866,-0.816032,0.758168,0
15to24Q-otAHmto7FzsWRg,-0.400175,0.564871,0.847454,-0.696757,0.130802,0.206345,-0.25532,0.103852,-0.296532,0.018263,-0.172969,-0.415172,-0.04715,-0.016866,1.225442,1.111536,1
8aqKdf4G4AAir8k_Kdslvg,-0.151622,0.564871,-1.053359,-0.306845,2.263476,-1.181509,-0.25532,0.642367,-0.213166,-1.279961,6.479669,-0.415172,-0.04715,-0.016866,-0.816032,-0.487166,0
uxU1vr5AhhkTQ83X0bpeyg,-0.400175,-1.770317,-1.204845,-0.134327,-1.312332,-0.939813,-0.25532,0.447907,-0.296532,0.018263,-0.172969,-0.415172,-0.04715,-0.016866,1.225442,0.757734,0
f702hTJoqdR34Jn23C7d1A,-0.400175,0.564871,-0.028287,-1.017614,-0.440124,-0.428198,-0.25532,0.046060,2.094199,-2.290786,-0.940257,-0.415172,-0.04715,-0.016866,1.225442,-2.647098,0


# 🧠 Treinamento dos Modelos

Com os dados preparados, passamos para a experimentação de diferentes arquiteturas de Machine Learning, buscando maximizar nossa métrica alvo: **Mean F1-Score ponderado**.

## 🌲 RandomForestClassifier
Modelo robusto baseado em ensemble de árvores de decisão. Utilizamos o `GridSearchCV` para encontrar os melhores hiperparâmetros mantendo a validação cruzada.

In [3]:
import treinamento_modelos as treinamento

score_treinamento, modelo_random_forest = treinamento.treinar_random_forest(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

INFO:root:Score RandomForestClassifier: 0.8477548019479663
INFO:root:Melhores parâmetros: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100, 'random_state': 42}


## Score treinamento: 0.8477548019479663

INFO:root:F1-score dataset validação: 0.8560459838986375


## Score validação: 0.8560459838986375

## 🚀 XGBoost Classifier
O *eXtreme Gradient Boosting* é conhecido por seu altíssimo desempenho em dados tabulares. Treinamento utilizando busca em grade iterativa focando em regularização.

In [4]:
score_treinamento, modelo_random_forest = treinamento.treinar_xgboost(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

C:\Users\rafae_69xt12o\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
INFO:root:Score XGBClassifier: 0.8477733880120436
INFO:root:Melhores parâmetros: {'colsample_bytree': 0.9, 'eval_metric': 'logloss', 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 100, 'random_state': 42, 'scale_pos_weight': 1, 'subsample': 0.8, 'use_label_encoder': False}
C:\Users\rafae_69xt12o\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


## Score treinamento: 0.8477733880120436

INFO:root:F1-score dataset validação: 0.8566024517673304


## Score validação: 0.8566024517673304

## 🕸️ Multi-Layer Perceptron (MLP)
Para complementar nossos modelos baseados em árvores, treinamos uma rede neural (MLP). Ajustamos a topologia das camadas ocultas e parâmetros de regularização.

In [ ]:
score_treinamento, modelo_random_forest = treinamento.treinar_mlp(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

## 🏆 Ensemble Final: Voting Classifier
Para extrair a máxima precisão de todos os estimadores, criamos um `VotingClassifier` (método *Hard Voting*). Esse ensemble unifica as predições do RandomForest, XGBoost e da Rede Neural (MLP).

In [6]:
score_treinamento, modelo_random_forest = treinamento.treinar_voting_classifier(df_treino)

display(Markdown(f"## Score treinamento: {score_treinamento}"))
display(
    Markdown(
        f"## Score validação: {treinamento.validar_modelo(modelo_random_forest, df_validacao)}"
    )
)

C:\Users\rafae_69xt12o\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
INFO:root:F1-Score do VotingClassifier: 0.8722995040270102


## Score treinamento: 0.8722995040270102

INFO:root:F1-score dataset validação: 0.8565005022034038


## Score validação: 0.8565005022034038